# 01 — Baseline: Logistic Regression

First working model for the return-risk scorer. Goal here is a defensible
baseline with honest, held-out metrics and a cost-aware threshold — not
peak accuracy. Later notebooks iterate on this.


## Load data and build the label

Using the English-translated CSVs (see `src/translate_dataset.py`).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.metrics import precision_recall_curve, roc_curve
import matplotlib.pyplot as plt

DATA_DIR = "../data"

orders = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv", parse_dates=[
    "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"])
items = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset_en.csv")
reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{DATA_DIR}/olist_products_dataset_en.csv")
customers = pd.read_csv(f"{DATA_DIR}/olist_customers_dataset_en.csv")

data = (orders
        .merge(items, on="order_id", how="left")
        .merge(payments, on="order_id", how="left")
        .merge(reviews[["order_id", "review_score"]], on="order_id", how="left")
        .merge(products, on="product_id", how="left")
        .merge(customers, on="customer_id", how="left"))

# Olist has no direct "returned" flag, so bad_order is a proxy: an order
# counts as bad if it was cancelled/unavailable, drew a low review score
# (<=2), or arrived more than 7 days late. review_score and cancellations
# are used as proxies for return/dissatisfaction risk since this mirrors
# how merchants without formal return tracking infer risk from reviews
# and cancellations.
data["delay_days"] = (data["order_delivered_customer_date"] - data["order_estimated_delivery_date"]).dt.days
data["bad_order"] = (
    (data["order_status"].isin(["cancelled", "unavailable"])) |
    (data["review_score"] <= 2) |
    (data["delay_days"] > 7)
).astype(int)

print("Positive rate:", data["bad_order"].mean())


## Feature engineering (v1)

In [ ]:
data["freight_ratio"] = data["freight_value"] / data["price"].replace(0, 1)
data["purchase_dow"] = data["order_purchase_timestamp"].dt.dayofweek
data["purchase_month"] = data["order_purchase_timestamp"].dt.month

num_features = ["price", "freight_value", "freight_ratio", "payment_installments",
                 "product_weight_g", "delay_days", "purchase_dow", "purchase_month"]
cat_features = ["payment_type", "product_category_name", "customer_state"]

data = data.dropna(subset=num_features + cat_features + ["bad_order"])
x = data[num_features + cat_features]
y = data["bad_order"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)


## Fit baseline logistic regression

`class_weight='balanced'` handles the ~15% positive rate; `StandardScaler` + `OneHotEncoder` are the standard preprocessing pair for mixed numeric/categorical logistic regression.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0))
])

model.fit(x_train[num_features + cat_features], y_train)
probs = model.predict_proba(x_test[num_features + cat_features])[:, 1]

print("AUC-ROC:", roc_auc_score(y_test, probs))
print("AUC-PR:", average_precision_score(y_test, probs))
print(classification_report(y_test, probs > 0.5))


## Cost-optimal threshold

A false positive (flagging a good order) is assumed to cost ₹50 in review
overhead; a false negative (missing a bad order) is assumed to cost ₹300
in unmanaged loss — a 6:1 asymmetry. The threshold is chosen to minimize
total cost on the test set, not to maximize any single classification
metric.


In [ ]:
cost_fp, cost_fn = 50, 300
thresholds = np.arange(0.1, 0.9, 0.01)
best_thresh, best_cost = None, float("inf")
for t in thresholds:
    preds = (probs > t).astype(int)
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    total_cost = fp * cost_fp + fn * cost_fn
    if total_cost < best_cost:
        best_cost, best_thresh = total_cost, t

print(f"Optimal threshold: {best_thresh}, cost: {best_cost}")
print(classification_report(y_test, probs > best_thresh))


## KS statistic

In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, probs)
ks_stat = max(tpr - fpr)
ks_threshold = roc_thresholds[np.argmax(tpr - fpr)]
print(f"KS Statistic: {ks_stat:.4f} at threshold {ks_threshold:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(roc_thresholds, tpr, label="True Positive Rate - Bad Orders Caught")
plt.plot(roc_thresholds, fpr, label="False Positive Rate - Good Orders Flagged")
plt.axvline(ks_threshold, color="gray", linestyle="--", label=f"Max separation at {ks_threshold:.2f}")
plt.gca().invert_xaxis()
plt.xlabel("Threshold")
plt.ylabel("Rate")
plt.title(f"KS Statistic = {ks_stat:.3f}")
plt.legend()
plt.tight_layout()
plt.show()


## Precision-recall curve

In [ ]:
precision, recall, pr_thresholds = precision_recall_curve(y_test, probs)
baseline = y_test.mean()

idx = np.argmin(np.abs(pr_thresholds - best_thresh))
chosen_precision, chosen_recall = precision[idx], recall[idx]

plt.figure(figsize=(6, 4))
plt.plot(recall, precision, label="Model")
plt.axhline(baseline, color="gray", linestyle="--", label=f"Baseline (positive rate = {baseline:.2f})")
plt.scatter([chosen_recall], [chosen_precision], color="red", zorder=5,
            label=f"Chosen Threshold ({pr_thresholds[idx]:.2f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()


## Result

AUC-ROC 0.673, AUC-PR 0.422, KS 28, cost-optimal threshold 0.55, cost
₹747,350. Precision at that threshold is only 0.25 — roughly 1 in 4 flagged
orders is a false positive. Given the 6:1 cost asymmetry, this is a
defensible trade-off, not obviously a bug, but it's worth understanding
*why* precision is capped this low before accepting it. Notebook 02
investigates via WOE/IV analysis, including a leakage check on `delay_days`,
which shows up as the single strongest predictor and is also a direct
component of the label.
